<a href="https://colab.research.google.com/github/shaikmujamil144/-Medical-drone-delivery/blob/main/MLLab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q mlflow joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 737.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.6/144.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import joblib
import mlflow
import mlflow.sklearn

import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
iris = load_iris()

X = iris.data
y = iris.target

print("Dataset shape:", X.shape)
print("Classes:", iris.target_names)

Dataset shape: (150, 4)
Classes: ['setosa' 'versicolor' 'virginica']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 120
Testing samples: 30


In [ ]:
mlflow.set_tracking_uri("sqlite:////content/mlflow.db")

mlflow.set_experiment("Model_Management_Experiment")

print("MLflow experiment created.")

2026/09/21 06:03:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/21 06:03:47 INFO mlflow.store.db.utils: Updating database tables
2026/09/21 06:03:49 INFO mlflow.tracking.fluent: Experiment with name 'Model_Management_Experiment' does not exist. Creating a new experiment.


MLflow experiment created.


In [ ]:
models = {
    "Logistic_Regression": LogisticRegression(max_iter=200),
    "Decision_Tree": DecisionTreeClassifier(random_state=42),
    "Random_Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

print("Models created:")
for name in models:
    print(name)

Models created:
Logistic_Regression
Decision_Tree
Random_Forest


In [ ]:
os.makedirs("/content/saved_models", exist_ok=True)

print("Model folder created.")

Model folder created.


In [ ]:
results = []

for name, model in models.items():

    with mlflow.start_run(run_name=name) as run:

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)

        model_file = f"/content/saved_models/{name}.pkl"
        joblib.dump(model, model_file)

        mlflow.log_param("model_name", name)
        mlflow.log_metric("accuracy", accuracy)

        mlflow.log_artifact(model_file)

        mlflow.sklearn.log_model(
          model,
          name="model",
          registered_model_name="Iris_Classifier",
          serialization_format="pickle"
        )

        results.append({
            "Model": name,
            "Accuracy": accuracy,
            "Run_ID": run.info.run_id,
            "Model_File": model_file
        })

        print(f"{name} -> Accuracy: {accuracy:.4f}")

2026/09/21 06:07:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Iris_Classifier' already exists. Creating a new version of this model...
Created version '2' of model 'Iris_Classifier'.
2026/09/21 06:07:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic_Regression -> Accuracy: 0.9667


Registered model 'Iris_Classifier' already exists. Creating a new version of this model...
Created version '3' of model 'Iris_Classifier'.


Decision_Tree -> Accuracy: 0.9333


2026/09/21 06:07:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random_Forest -> Accuracy: 0.9000


Registered model 'Iris_Classifier' already exists. Creating a new version of this model...
Created version '4' of model 'Iris_Classifier'.


In [ ]:
client = mlflow.MlflowClient()

versions = client.search_model_versions(
    "name='Iris_Classifier'"
)

all_results = []

for version in versions:
    run = client.get_run(version.run_id)

    accuracy = run.data.metrics.get("accuracy")
    model_name = run.data.params.get("model_name")

    if accuracy is not None and model_name is not None:
        all_results.append({
            "Model": model_name,
            "Accuracy": accuracy,
            "Version": version.version,
            "Run_ID": version.run_id
        })

results_df = pd.DataFrame(all_results)

print(results_df)

                 Model  Accuracy  Version                            Run_ID
0        Random_Forest  0.900000        4  7a6450d7e5f7456e9a0b7c38de953ba4
1        Decision_Tree  0.933333        3  309b7d1b37cf4c56a4f92bcb28d535b8
2  Logistic_Regression  0.966667        2  da260b6f067a49dcbdf85772bd5c0532
3  Logistic_Regression  0.966667        1  70e32cbdb5004e35bf6dcf619374cfc8


In [ ]:
best_model_row = results_df.loc[
    results_df["Accuracy"].idxmax()
]

best_model_name = best_model_row["Model"]
best_accuracy = best_model_row["Accuracy"]
best_version = best_model_row["Version"]

print("Best Model:", best_model_name)
print("Accuracy:", best_accuracy)
print("MLflow Version:", best_version)

Best Model: Logistic_Regression
Accuracy: 0.9666666666666667
MLflow Version: 2


In [ ]:
best_model_row = results_df.loc[
    results_df["Accuracy"].idxmax()
]

best_model_name = best_model_row["Model"]
best_accuracy = best_model_row["Accuracy"]
best_version = best_model_row["Version"]

print("Best Model:", best_model_name)
print("Accuracy:", best_accuracy)
print("MLflow Version:", best_version)

Best Model: Logistic_Regression
Accuracy: 0.9666666666666667
MLflow Version: 2


In [ ]:
client = mlflow.MlflowClient()

versions = client.search_model_versions(
    "name='Iris_Classifier'"
)

all_results = []

for version in versions:
    run = client.get_run(version.run_id)

    accuracy = run.data.metrics.get("accuracy")
    model_name = run.data.params.get("model_name")

    if accuracy is not None and model_name is not None:
        all_results.append({
            "Model": model_name,
            "Accuracy": accuracy,
            "Version": version.version
        })

results_df = pd.DataFrame(all_results)

print(results_df)

                 Model  Accuracy  Version
0        Random_Forest  0.900000        4
1        Decision_Tree  0.933333        3
2  Logistic_Regression  0.966667        2
3  Logistic_Regression  0.966667        1


In [ ]:
best_model_row = results_df.loc[
    results_df["Accuracy"].idxmax()
]

best_model_name = best_model_row["Model"]
best_accuracy = best_model_row["Accuracy"]
best_version = best_model_row["Version"]

print("Best Model:", best_model_name)
print("Best Accuracy:", best_accuracy)
print("Registered Version:", best_version)

Best Model: Logistic_Regression
Best Accuracy: 0.9666666666666667
Registered Version: 2


In [ ]:
model_uri = f"models:/Iris_Classifier/{best_version}"

registered_model = mlflow.sklearn.load_model(model_uri)

print("Best model loaded successfully.")

Best model loaded successfully.


In [ ]:
new_data = pd.DataFrame(
    [
        [5.1, 3.5, 1.4, 0.2],
        [6.2, 2.8, 4.8, 1.8],
        [6.7, 3.0, 5.2, 2.3]
    ],
    columns=iris.feature_names
)

print(new_data)

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                6.2               2.8                4.8               1.8
2                6.7               3.0                5.2               2.3


In [ ]:
predictions = registered_model.predict(new_data)

print("Predictions:")

for i, prediction in enumerate(predictions):
    print(
        f"Sample {i + 1}: {iris.target_names[prediction]}"
    )

Predictions:
Sample 1: setosa
Sample 2: virginica
Sample 3: virginica


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [ ]:
print("========================================")
print("       MODEL MANAGEMENT SUMMARY")
print("========================================")

print("Registered Model : Iris_Classifier")
print("Best Model       :", best_model_name)
print("Best Accuracy    :", best_accuracy)
print("Model Version    :", best_version)

print("\nPredictions on New Data:")

for i, prediction in enumerate(predictions):
    print(
        f"Sample {i + 1}: {iris.target_names[prediction]}"
    )

print("\nExperiment completed successfully.")

       MODEL MANAGEMENT SUMMARY
Registered Model : Iris_Classifier
Best Model       : Logistic_Regression
Best Accuracy    : 0.9666666666666667
Model Version    : 2

Predictions on New Data:
Sample 1: setosa
Sample 2: virginica
Sample 3: virginica

Experiment completed successfully.
